<a href="https://colab.research.google.com/github/crosoriom/IntroduccionAprendizajeMaquina/blob/main/DashBoard_Parcial_1_TAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install necessary packages
!pip install dash dash-bootstrap-components pyngrok

# Import libraries
import dash
from dash import dcc, html, Input, Output, dash_table
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import warnings
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from pyngrok import ngrok
import dash_bootstrap_components as dbc

# The RenameDatabase import needs to be handled differently in Colab
# Define the function inline instead of importing
def renameDatabase(df):
    df = df.copy()  # Create a copy to avoid modifying the original
    column_mapping = {
        'PName': 'Player_Name',
        'POS': 'Position',
        'GP': 'Games_Played',
        'W': 'Wins',
        'L': 'Loses',
        'Min': 'Minutes_Played',
        'PTS': 'Total_Points',
        'FGM': 'Field_Goals_Made',
        'FGA': 'Field_Goals_Attempted',
        'FG%': 'Field_Goal_Percentage',
        '3PM': 'Three_Point_FG_Made',
        '3PA': 'Three_Point_FG_Attempted',
        '3P%': 'Three_Point_FG_Percentage',
        'FTM': 'Free_Throws_Made',
        'FTA': 'Free_Throws_Attempted',
        'FT%': 'Free_Throws_Percentage',
        'OREB': 'Offensive_Rebounds',
        'DREB': 'Defensive_Rebounds',
        'REB': 'Total_Rebounds',
        'AST': 'Assists',
        'TOV': 'Turnovers',
        'STL': 'Steals',
        'BLK': 'Blocks',
        'PF': 'Personal_Fouls',
        'FP': 'NBA_Fantasy_Points',
        'DD2': 'Double_Doubles',
        'TD3': 'Triple_Doubles',
        '+/-': 'Plus_Minus'
    }
    df.rename(columns=column_mapping, inplace=True)
    return df

# Ignore warnings
warnings.filterwarnings('ignore')

# For Colab, you need to upload your data files or get them from a URL
# Let's handle both cases

# Option 1: Upload files to Colab session
from google.colab import files
print("Please upload your 2023_nba_player_stats.csv file:")
uploaded = files.upload()

# Option 2: Or use sample data if files aren't uploaded
if '2023_nba_player_stats.csv' not in uploaded:
    # Create sample data for demonstration
    print("Using sample NBA data for demonstration")
    import numpy as np

    # Create sample player data
    np.random.seed(42)
    n_players = 100

    sample_df = pd.DataFrame({
        'PName': [f'Player_{i}' for i in range(n_players)],
        'POS': np.random.choice(['PG', 'SG', 'SF', 'PF', 'C'], n_players),
        'Age': np.random.randint(19, 40, n_players),
        'Team': np.random.choice(['LAL', 'BOS', 'GSW', 'MIA', 'CHI'], n_players),
        'GP': np.random.randint(20, 82, n_players),
        'W': np.random.randint(10, 60, n_players),
        'L': np.random.randint(10, 40, n_players),
        'Min': np.random.randint(500, 3000, n_players),
        'PTS': np.random.randint(100, 2500, n_players),
        'FGM': np.random.randint(50, 900, n_players),
        'FGA': np.random.randint(100, 1800, n_players),
        'FG%': np.random.uniform(0.3, 0.6, n_players),
        '3PM': np.random.randint(0, 300, n_players),
        '3PA': np.random.randint(0, 800, n_players),
        '3P%': np.random.uniform(0.25, 0.45, n_players),
        'FTM': np.random.randint(20, 500, n_players),
        'FTA': np.random.randint(30, 600, n_players),
        'FT%': np.random.uniform(0.6, 0.95, n_players),
        'OREB': np.random.randint(10, 300, n_players),
        'DREB': np.random.randint(30, 700, n_players),
        'REB': np.random.randint(50, 1000, n_players),
        'AST': np.random.randint(20, 800, n_players),
        'TOV': np.random.randint(20, 300, n_players),
        'STL': np.random.randint(10, 200, n_players),
        'BLK': np.random.randint(0, 200, n_players),
        'PF': np.random.randint(30, 300, n_players),
        'FP': np.random.randint(200, 3000, n_players),
        'DD2': np.random.randint(0, 40, n_players),
        'TD3': np.random.randint(0, 10, n_players),
        '+/-': np.random.randint(-300, 300, n_players)
    })

    sample_df.to_csv('2023_nba_player_stats.csv', index=False)

# Load and process data
df = pd.read_csv('2023_nba_player_stats.csv')
df = renameDatabase(df)

# Create processed database.csv
columns_drop = ['Player_Name', 'Team', 'Position', 'Field_Goals_Made', 'Field_Goal_Percentage',
                'Three_Point_FG_Made', 'Three_Point_FG_Percentage', 'Free_Throws_Made',
                'Free_Throws_Percentage', 'Wins', 'Loses', 'Defensive_Rebounds', 'Offensive_Rebounds',
                'Blocks', 'Double_Doubles', 'Triple_Doubles']

model_data = df.drop(columns=columns_drop, errors='ignore')
model_data.dropna(inplace=True)
model_data.to_csv('database.csv', index=False)

# Create sample model results if not available
try:
    print("Please upload your Model comparison results.csv file:")
    uploaded = files.upload()
    results_df = pd.read_csv('Model comparison results.csv')
except:
    print("Creating sample model results")
    models = ['LinearRegression', 'Lasso', 'ElasticNet', 'Ridge', 'KernelRidge',
              'SGDRegressor', 'BayesianRidge', 'GaussianProcess', 'RandomForest', 'SVR']

    results_df = pd.DataFrame({
        'Model': models,
        'Test_MSE': np.random.uniform(1000, 5000, len(models)),
        'Test_MAE': np.random.uniform(20, 50, len(models)),
        'Test_R2': np.random.uniform(0.7, 0.95, len(models)),
        'Test_MAPE': np.random.uniform(0.05, 0.15, len(models)),
        'Training_Time': np.random.uniform(0.5, 10, len(models)),
        'Best_MSE_CV': np.random.uniform(1100, 5500, len(models)),
        'Best_MAE_CV': np.random.uniform(22, 55, len(models))
    })

    results_df.to_csv('Model comparison results.csv', index=False)

# Create dashboard app with Bootstrap styling
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Layout for the dashboard
app.layout = dbc.Container([
    html.H1("NBA Player Stats Analysis Dashboard",
            style={'textAlign': 'center', 'color': '#1D428A', 'marginBottom': 30, 'marginTop': 20}),

    dcc.Tabs([
        # Data Overview Tab
        dcc.Tab(label='Data Overview', children=[
            dbc.Row([
                dbc.Col([
                    html.H3("NBA Player Stats Overview", style={'margin': '20px 0px'}),

                    html.Div([
                        html.H4("Dataset Statistics"),
                        dash_table.DataTable(
                            data=df.describe().reset_index().round(2).to_dict('records'),
                            columns=[{"name": i, "id": i} for i in df.describe().reset_index().columns],
                            style_table={'overflowX': 'auto'},
                            style_cell={'textAlign': 'center', 'padding': '10px'},
                            style_header={'backgroundColor': '#1D428A', 'color': 'white', 'fontWeight': 'bold'}
                        )
                    ], style={'width': '100%', 'margin': '10px'}),

                    html.Div([
                        html.H4("Data Sample"),
                        dash_table.DataTable(
                            data=df.head(10).to_dict('records'),
                            columns=[{"name": i, "id": i} for i in df.columns],
                            style_table={'overflowX': 'auto', 'maxHeight': '400px'},
                            style_cell={'textAlign': 'center', 'padding': '5px'},
                            style_header={'backgroundColor': '#1D428A', 'color': 'white', 'fontWeight': 'bold'}
                        )
                    ], style={'width': '100%', 'margin': '10px'}),
                ])
            ])
        ]),

        # Player Stats Analysis Tab
        dcc.Tab(label='Player Stats Analysis', children=[
            dbc.Row([
                dbc.Col([
                    html.H3("Player Stats Analysis", style={'margin': '20px 0px'}),

                    dbc.Row([
                        dbc.Col([
                            html.H4("Points Distribution"),
                            dcc.Graph(
                                figure=px.histogram(df, x='Total_Points',
                                                  title='Distribution of Total Points',
                                                  color_discrete_sequence=['#C8102E'])
                            )
                        ], width=6),

                        dbc.Col([
                            html.H4("Points per Position"),
                            dcc.Graph(
                                figure=px.box(df, x='Position', y='Total_Points',
                                             title='Points Distribution by Position',
                                             color='Position')
                            )
                        ], width=6),
                    ]),

                    dbc.Row([
                        dbc.Col([
                            html.H4("Points vs Age"),
                            dcc.Graph(
                                figure=px.scatter(df, x='Age', y='Total_Points',
                                                 color='Position',
                                                 title='Age vs Total Points',
                                                 trendline='ols')
                            )
                        ], width=6),

                        dbc.Col([
                            html.H4("Correlation Heatmap"),
                            dcc.Graph(
                                figure=px.imshow(model_data.corr(),
                                                title='Feature Correlation Matrix',
                                                color_continuous_scale='RdBu_r')
                            )
                        ], width=6),
                    ]),

                    dbc.Row([
                        dbc.Col([
                            html.H4("Feature Explorer"),
                            html.P("Select a feature to analyze its relationship with Total Points:"),
                            dcc.Dropdown(
                                id='feature-dropdown',
                                options=[{'label': col, 'value': col} for col in model_data.columns if col != 'Total_Points'],
                                value='Field_Goals_Attempted',
                                style={'width': '50%'}
                            ),
                            dcc.Graph(id='feature-points-scatter')
                        ])
                    ])
                ])
            ])
        ]),

        # Model Performance Tab
        dcc.Tab(label='Model Performance', children=[
            dbc.Row([
                dbc.Col([
                    html.H3("Model Performance Analysis", style={'margin': '20px 0px'}),

                    dbc.Row([
                        dbc.Col([
                            html.H4("Error Metrics Comparison"),
                            dcc.Graph(
                                figure=px.bar(results_df, x='Model', y=['Test_MSE', 'Test_MAE'],
                                             barmode='group',
                                             title='Error Metrics by Model',
                                             color_discrete_sequence=['#C8102E', '#1D428A'])
                            )
                        ], width=6),

                        dbc.Col([
                            html.H4("R² Score by Model"),
                            dcc.Graph(
                                figure=px.bar(results_df, x='Model', y='Test_R2',
                                             title='R² Score by Model',
                                             color='Test_R2',
                                             color_continuous_scale='viridis')
                            )
                        ], width=6),
                    ]),

                    dbc.Row([
                        dbc.Col([
                            html.H4("Performance vs Training Time"),
                            dcc.Graph(
                                figure=px.scatter(results_df, x='Training_Time', y='Test_R2',
                                                 size='Test_MSE', size_max=50,
                                                 hover_name='Model',
                                                 title='Model Performance vs Training Time',
                                                 labels={'Training_Time': 'Training Time (s)', 'Test_R2': 'R² Score'},
                                                 color='Model')
                            )
                        ])
                    ]),

                    dbc.Row([
                        dbc.Col([
                            html.H4("Model Metrics Details"),
                            dash_table.DataTable(
                                data=results_df.round(4).to_dict('records'),
                                columns=[{"name": i, "id": i} for i in results_df.columns],
                                style_table={'overflowX': 'auto'},
                                style_cell={'textAlign': 'center', 'padding': '10px'},
                                style_header={'backgroundColor': '#1D428A', 'color': 'white', 'fontWeight': 'bold'},
                                sort_action='native',
                                filter_action='native'
                            )
                        ])
                    ])
                ])
            ])
        ]),

        # Predictions Tab
        dcc.Tab(label='Predict Points', children=[
            dbc.Row([
                dbc.Col([
                    html.H3("Player Points Predictor", style={'margin': '20px 0px'}),

                    dbc.Card([
                        dbc.CardHeader(html.H4("Input Player Stats")),
                        dbc.CardBody([
                            dbc.Row([
                                dbc.Col([
                                    html.Label("Minutes Played:"),
                                    dcc.Input(id='minutes-input', type='number', value=25, style={'width': '100%'})
                                ], width=4),
                                dbc.Col([
                                    html.Label("Field Goals Attempted:"),
                                    dcc.Input(id='fga-input', type='number', value=15, style={'width': '100%'})
                                ], width=4),
                                dbc.Col([
                                    html.Label("Three Point Attempts:"),
                                    dcc.Input(id='3pa-input', type='number', value=5, style={'width': '100%'})
                                ], width=4)
                            ], className="mb-3"),

                            dbc.Row([
                                dbc.Col([
                                    html.Label("Free Throws Attempted:"),
                                    dcc.Input(id='fta-input', type='number', value=4, style={'width': '100%'})
                                ], width=4),
                                dbc.Col([
                                    html.Label("Assists:"),
                                    dcc.Input(id='ast-input', type='number', value=3, style={'width': '100%'})
                                ], width=4),
                                dbc.Col([
                                    html.Label("Steals:"),
                                    dcc.Input(id='stl-input', type='number', value=1, style={'width': '100%'})
                                ], width=4)
                            ], className="mb-3"),

                            dbc.Row([
                                dbc.Col([
                                    html.Label("Turnovers:"),
                                    dcc.Input(id='tov-input', type='number', value=2, style={'width': '100%'})
                                ], width=4),
                                dbc.Col([
                                    html.Label("Total Rebounds:"),
                                    dcc.Input(id='reb-input', type='number', value=5, style={'width': '100%'})
                                ], width=4),
                                dbc.Col([
                                    html.Label("Personal Fouls:"),
                                    dcc.Input(id='pf-input', type='number', value=2, style={'width': '100%'})
                                ], width=4)
                            ], className="mb-3"),

                            dbc.Button('Predict Points', id='predict-button',
                                      color='danger', className='mt-3'),

                            html.Div(id='prediction-output',
                                    style={'margin': '20px 0', 'fontWeight': 'bold', 'fontSize': '18px'})
                        ])
                    ], style={'marginBottom': '20px'}),

                    html.Div([
                        html.H4("Prediction Comparison"),
                        dcc.Graph(id='prediction-comparison')
                    ])
                ])
            ])
        ])
    ], style={'marginTop': '20px'})
], fluid=True)

# Callback for feature-points scatter plot
@app.callback(
    Output('feature-points-scatter', 'figure'),
    [Input('feature-dropdown', 'value')]
)
def update_feature_scatter(feature):
    fig = px.scatter(df, x=feature, y='Total_Points',
                    color='Position',
                    title=f'{feature} vs Total Points',
                    trendline='ols')
    return fig

# Prepare a simple model for predictions
@app.callback(
    [Output('prediction-output', 'children'),
     Output('prediction-comparison', 'figure')],
    [Input('predict-button', 'click')],
    [dash.dependencies.State('minutes-input', 'value'),
     dash.dependencies.State('fga-input', 'value'),
     dash.dependencies.State('3pa-input', 'value'),
     dash.dependencies.State('fta-input', 'value'),
     dash.dependencies.State('ast-input', 'value'),
     dash.dependencies.State('stl-input', 'value'),
     dash.dependencies.State('tov-input', 'value'),
     dash.dependencies.State('reb-input', 'value'),
     dash.dependencies.State('pf-input', 'value')]
)
def predict_points(n_clicks, minutes, fga, tpa, fta, ast, stl, tov, reb, pf):
    # Prepare the model data
    X = model_data.drop('Total_Points', axis=1)
    y = model_data['Total_Points']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train three models for comparison
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
    }

    # Handle missing columns in input data
    required_columns = X.columns.tolist()

    # Create input data dictionary matching feature names
    input_data_dict = {
        'Minutes_Played': [minutes],
        'Field_Goals_Attempted': [fga],
        'Three_Point_FG_Attempted': [tpa],
        'Free_Throws_Attempted': [fta],
        'Total_Rebounds': [reb],
        'Assists': [ast],
        'Turnovers': [tov],
        'Steals': [stl],
        'Personal_Fouls': [pf]
    }

    # Add any missing columns with zeros
    for col in required_columns:
        if col not in input_data_dict:
            input_data_dict[col] = [0]

    input_data = pd.DataFrame(input_data_dict)

    # Make sure input_data has all the required columns in the right order
    input_data = input_data.reindex(columns=required_columns, fill_value=0)

    # Make predictions with each model
    predictions = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        predictions[name] = model.predict(input_data)[0]

    # Create comparison figure
    fig = go.Figure()
    for model_name, pred in predictions.items():
        fig.add_trace(go.Bar(
            x=[model_name],
            y=[pred],
            name=model_name
        ))

    fig.update_layout(
        title='Predicted Points by Different Models',
        yaxis_title='Predicted Points',
        barmode='group'
    )

    # Round the prediction from the best model (Random Forest)
    rf_prediction = round(predictions['Random Forest'], 1)

    return f"Predicted Points: {rf_prediction}", fig

# Set up Ngrok to create a public URL
!ngrok authtoken
public_url = ngrok.connect(8050)
print(f"Dashboard is publicly accessible at: {public_url}")

# Run the app
if __name__ == '__main__':
    app.run(debug=True, port=8050)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.9/202.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 13.4 MB/s eta 0:00:00
  Attempting uninstall: Werkzeug
    Found existing installation: Werkzeug 3.1.3
    Uninstalling Werkzeug-3.1.3:
      Successfully uninstalled Werkzeug-3.1.3
  Attempting uninstall: Flask
    Found existing installation: Flask 3.1.0
    Uninstalling Flask-3.1.0:
      Successfully uninstalled Flask-3.1.0
Please upload your 2023_nba_player_stats.csv file:


Saving 2023_nba_player_stats.csv to 2023_nba_player_stats.csv
Please upload your Model comparison results.csv file:


Saving Model comparison results.csv to Model comparison results.csv
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Dashboard is publicly accessible at: NgrokTunnel: "https://f395-34-23-112-74.ngrok-free.app" -> "http://localhost:8050"


<IPython.core.display.Javascript object>